## WALLOC Phase-1 gate on Kaggle 2xGPU

Trains the 7-run gate matrix (PD vs SD across perception weights + a deterministic ablation), evaluates each on Kodak, and decides pass/kill against the gate criteria from the planning doc.

**Before running:**
- Settings → Accelerator → **GPU T4 × 2**
- Settings → Internet → **On**
- Add Data → attach a training image folder (DIV2K / CLIC2020) and the Kodak dataset

Outputs land in `/kaggle/working/walloc_gate/`. Re-running this notebook skips already-finished runs.

In [ ]:
!git clone https://github.com/KhoiTrant68/lattice-transform-coding.git /kaggle/working/walloc
%cd /kaggle/working/walloc

In [ ]:
%%bash
# Detect attached datasets (the names below match common Kaggle uploads;
# adjust if yours differ).
ls /kaggle/input
TRAIN=$(ls -d /kaggle/input/*div2k* 2>/dev/null | head -1)
TEST=$(ls -d  /kaggle/input/*kodak* 2>/dev/null | head -1)
echo "TRAIN=$TRAIN"
echo "TEST=$TEST"

bash setup.sh \
    --train-input "$TRAIN" \
    --test-input  "$TEST" \
    --output      /kaggle/working/walloc_gate \
    --gpus        all

In [ ]:
import os, subprocess, torch

# Load env into this Python process.
for line in subprocess.check_output(["bash", "-c", "source .walloc_env && env"]).decode().splitlines():
    if "=" in line and line.split("=", 1)[0].startswith(("WALLOC_", "CUDA_", "WANDB_", "PYTHONPATH")):
        k, v = line.split("=", 1)
        os.environ[k] = v

assert torch.cuda.is_available(), "CUDA not available — change accelerator to GPU T4 × 2"
print("GPUs visible:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB")
for k in ("WALLOC_TRAIN_INPUT", "WALLOC_TEST_INPUT", "WALLOC_OUTPUT_DIR", "WALLOC_NUM_GPUS"):
    print(f"  {k}={os.environ.get(k)}")

## Sanity tests (~3 min)

Each test maps to a load-bearing piece of theory (crypto lemma, Prop 3.3, QSD volume correction, ...). If anything here fails, FIX BEFORE training.

In [ ]:
!pip install -q pytest
!pytest tests/test_walloc.py -v --tb=short

## Run the gate matrix (~6-8 h)

Sequential: 7 trains + 7 evals. Re-running this cell after a timeout picks up where it left off (per-run `summary.json` + `eval_kodak.json` are the skip markers).

In [ ]:
%%bash
source .walloc_env
python -u scripts/run_gate.py \
    --out_root  "$WALLOC_OUTPUT_DIR" \
    --epochs 20 --batch_size 16 --patch_size 256 \
    --channels 128 --N_integral 512 --num_workers 2 \
    --ntc_quality 4 --lmbda 0.013

## TensorBoard

Per-run scalars (loss / bpp / psnr / perc / aux / lr) and per-epoch images (input / recon / |residual|) land under `$WALLOC_OUTPUT_DIR/<tag>/tb/` during training, and per-image eval scalars + recon previews under `<tag>/tb_eval/` after `scripts/eval.py` runs. The cell below mounts all of them.

In [ ]:
%load_ext tensorboard
import os
%tensorboard --logdir {os.environ["WALLOC_OUTPUT_DIR"]} --port 6006

## Inspect results

In [ ]:
import json, os
import pandas as pd
with open(os.path.join(os.environ["WALLOC_OUTPUT_DIR"], "GATE_RESULTS.json")) as f:
    g = json.load(f)
df = pd.DataFrame(g["table"])
if not df.empty:
    cols = ["tag", "mode", "lmbda", "lmbda_p", "s_dither",
            "bpp_mean", "psnr_mean", "sw2_patch_mean"]
    if "lpips_mean" in df.columns:
        cols.append("lpips_mean")
    display(df[cols].sort_values(["mode", "lmbda_p"]))
print("\nGate decision:")
for k, v in g["gate"].items():
    if k != "notes":
        print(f"  {k}: {v}")
print("\nNotes:")
for n in g["gate"]["notes"]:
    print(f"  - {n}")

In [ ]:
# Quick R-D-P scatter
import matplotlib.pyplot as plt
if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for mode, sub in df.groupby('mode'):
        axes[0].scatter(sub['bpp_mean'], sub['psnr_mean'], s=80, label=mode)
        axes[1].scatter(sub['bpp_mean'], sub['sw2_patch_mean'], s=80, label=mode)
    axes[0].set(xlabel='bpp', ylabel='PSNR (dB)', title='R-D')
    axes[1].set(xlabel='bpp', ylabel='SW2 patch (lower = better)', title='R-P')
    axes[1].set_yscale('log')
    for a in axes:
        a.legend(); a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()